# Construct VOCAB table

*Please note that dfidf gets added to VOCAB in 04-bow-tfidf.ipynb*

## Setup

### Import Libraries

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import glob
import os
import re
import nltk
import xml.etree.ElementTree as ET
from pathlib import Path
from bs4 import BeautifulSoup
from nltk.stem.porter import PorterStemmer

In [ ]:
nltk_resources = [
    'punkt_tab',
    'averaged_perceptron_tagger_eng',
    'stopwords',
    'tagsets'
]

for resource in nltk_resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Could not download {resource}: {e}")

### Configuration and Data Loading

In [ ]:
# declare OHCO and load in the corpus
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
CORPUS = pd.read_csv('data/CORPUS.csv', sep='\t').set_index(OHCO)

## Extract VOCAB

Features:
- n, p, i, n_chars

In [ ]:
# extract VOCAB from CORPUS and get counts
VOCAB = CORPUS.term_str.value_counts().to_frame('n').sort_index()

# derive number of characters in terms
VOCAB['n_chars'] = VOCAB.index.str.len()

# derive probability (likelihood of a token appearing in the text)
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()

# derive information (how surprising or informative a term is)
VOCAB['i'] = -np.log2(VOCAB.p)

## Annotate VOCAB

Features:
- max_pos, max_pos_group
- n_pos, n_pos_group (pos ambiguity)
- stop (stopwords)
- porter stem
- dfidf (gets added later in pipeline)
- add ngrams if time permits?

In [ ]:
# get max POS (most frequently associated part of speech category for each word)

# max pos per term (count every pair, unstack pos tags into columns, find column name with highest count per row)
VOCAB['max_pos'] = CORPUS[['term_str', 'pos']].value_counts().unstack(fill_value=0).idxmax(1)

# assign term a max pos group
VOCAB['max_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().unstack(fill_value=0).idxmax(1)

In [ ]:
# compute POS ambiguity (how many pos, by group, are associated with each term in the corpus)

# 
VOCAB['n_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().unstack().count(1)
# VOCAB['cat_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().to_frame('n').reset_index()\
#     .groupby('term_str').pos_group.apply(lambda x: set(x))
VOCAB['n_pos'] = CORPUS[['term_str','pos']].value_counts().unstack().count(1)
# VOCAB['cat_pos'] = CORPUS[['term_str','pos']].value_counts().to_frame('n').reset_index()\
#     .groupby('term_str').pos.apply(lambda x: set(x))

# TODO FIX REDUNDANT COMPUTATION?

In [ ]:
VOCAB.sample(10)

In [ ]:
VOCAB.loc[['s','t','don','ve','ll']]

## Add Stopwords

Using NLTK's built in list for English and choosing to add 'said' to stopword list as it functions mostly as a dialogue marker rather than a content-bearing term. I am also adding contraction fragments.

In [ ]:
# get stopwords from NLTK's built in list for English
sw = set(nltk.corpus.stopwords.words('english'))
# add dialogue marker to stopwords
# sw.update(['said']) # use .update() to add elements from an iterable into an existing set in place
sw.update(['said', 'couldn', 'didn', 'wouldn', 'shouldn', 'isn', 'aren', 'weren', 'haven', 'hadn', 'doesn', 'won', 't', 're', 've', 'll', 'd', 'm', 's']) # use .update() to add elements from an iterable into an existing set in place
VOCAB['stop'] = VOCAB.index.isin(sw)

In [ ]:
fragments = ['t', 're', 've', 'll', 'd', 'm', 's', 'couldn', 'didn', 'wouldn', 'shouldn', 'isn', 'aren', 'weren', 'haven', 'hadn', 'doesn', 'won']
VOCAB[VOCAB.index.isin(fragments)]

In [ ]:
# check high freq terms not flagged as stopwords
VOCAB[VOCAB.stop == False].sort_values('n', ascending=False).head(50)

In [ ]:
# check flagged stopwords for things that shouldn't be in stopwords

In [ ]:
VOCAB[VOCAB.stop == True].sort_values('n', ascending=False).head(30)

In [ ]:
VOCAB[VOCAB.stop == 1]

In [ ]:
print("Stopwords:", int(VOCAB.stop.sum()))

## Add Porter Stem

In [ ]:
# instantiate stemmer (outside the lambda function)
porter = PorterStemmer()

VOCAB['porter_stem'] = VOCAB.index.map(lambda x: porter.stem(x))

In [ ]:
VOCAB

In [ ]:
# check
VOCAB.columns.tolist()

## Save Output

In [ ]:
# save the VOCAB table to csv
VOCAB.to_csv('data/VOCAB.csv', sep='\t', index=True)